In [1]:
# Colab setup - run this cell first
import sys, os
from pathlib import Path

if 'google.colab' in sys.modules:
    repo_dir = '/content/CompMathAndAICourse/'
    if not os.path.exists(repo_dir):
        !git clone https://github.com/lruthotto/CompMathAndAICourse.git {repo_dir}
        %pip install -q jax jaxlib jaxopt h5py

    NOTEBOOK_DIR = Path(repo_dir) / 'code' / '07-sciml'
    sys.path.insert(0, str(NOTEBOOK_DIR))
    print(f"Running on Colab, repo cloned to {repo_dir}")
else:
    # Local execution: get notebook directory from VS Code or fallback
    try:
        import IPython
        notebook_path = IPython.extract_module_locals()[1]['__vsc_ipynb_file__']
        NOTEBOOK_DIR = Path(notebook_path).parent
    except (KeyError, AttributeError, TypeError):
        # Fallback: search for utils.py to find the correct directory
        for candidate in [Path.cwd(), Path.cwd() / 'code' / '07-sciml']:
            if (candidate / 'utils.py').exists():
                NOTEBOOK_DIR = candidate
                break
        else:
            NOTEBOOK_DIR = Path.cwd()
    
    if str(NOTEBOOK_DIR) not in sys.path:
        sys.path.insert(0, str(NOTEBOOK_DIR))
    print(f"Running locally from {NOTEBOOK_DIR}")

Running locally from /workspace/code/07-sciml


# Physics-Informed Neural Networks (PINNs) for Darcy Flow

**Course:** Computational Mathematics and AI  
**Lecture 7:** Scientific ML for PDEs

## Overview

This notebook demonstrates **Physics-Informed Neural Networks (PINNs)** for solving the Darcy flow equation:

$$-\nabla \cdot (\kappa(x,y) \nabla u) = f \quad \text{in } \Omega = [0,1]^2$$
$$u = 0 \quad \text{on } \partial\Omega$$

### Key Idea

PINNs use a neural network to approximate the solution $u_\theta(x,y)$ and train by minimizing:

$$\mathcal{L}(\theta) = \lambda_{\text{PDE}} \underbrace{\| -\nabla \cdot (\kappa \nabla u_\theta) - f \|^2_{\text{domain}}}_{\text{PDE residual}} + \lambda_{\text{BC}} \underbrace{\| u_\theta \|^2_{\text{boundary}}}_{\text{BC loss}}$$

**No training data needed!** The PDE physics serves as the supervisory signal.

## Learning Objectives

1. Understand how PINNs encode PDE constraints via automatic differentiation
2. Implement PINN training with loss balancing
3. Observe PINN limitations (spectral bias, high-frequency challenges)

In [2]:
import jax
import jax.numpy as jnp
from jax import random, grad, jit, vmap
import numpy as np
from time import time
import pandas as pd

# ============================================================
# Display Configuration
# ============================================================
# Set INTERACTIVE = True for interactive plotting (Jupyter/Colab)
# Set INTERACTIVE = False for headless environments (saves figures only)
INTERACTIVE = False

import matplotlib
if INTERACTIVE:
    matplotlib.use('module://matplotlib_inline.backend_inline')
else:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt

def savefig(fig, path, **kwargs):
    """Save figure and optionally display it."""
    kwargs.setdefault('dpi', 150)
    kwargs.setdefault('bbox_inches', 'tight')
    fig.savefig(path, **kwargs)
    print(f"Saved: {path}")
    if INTERACTIVE:
        plt.show()
    else:
        plt.close(fig)
# ============================================================

from utils import download_pdebench, compute_l2_error, plot_solution_comparison, plot_convergence, save_individual_figure
from models import initialize_mlp, mlp_forward, count_params

# Set up JAX
print(f"JAX devices: {jax.devices()}")
print(f"JAX version: {jax.__version__}")

# Create output directory for figures (relative to notebook location)
OUTPUT_DIR = NOTEBOOK_DIR / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# Master PRNG key for reproducibility
MASTER_KEY = random.PRNGKey(42)

JAX devices: [CpuDevice(id=0)]
JAX version: 0.8.1
Output directory: /workspace/code/07-sciml/figures


## Configuration

We use **pure Adam optimization** with 20,000 iterations and random sampling of collocation points.

This approach:
- Uses mini-batch stochastic gradient descent with random interior/boundary points
- Avoids the complexity and computational cost of L-BFGS
- Matches the training time (~300s) shown in lecture slides

In [3]:
# Configuration for pure Adam training (matching slides: 20k iterations, ~300s)
config = {
    # Network
    'hidden_layers': [32, 32, 32, 32],
    'activation': 'gelu',

    # Problem
    'resolution': 128,  # Kappa field resolution
    'kappa_index': 0,  # Which PDEBench sample to use

    # Sampling (for Adam phase)
    'n_interior': 2048,    # Interior points per batch
    'n_boundary': 512,     # Boundary points per batch

    # Loss weights
    'lambda_pde': 1.0,
    'lambda_bc': 100.0,  # Weight for boundary conditions

    # Training: Pure Adam (20k iterations to match slides)
    'adam_lr': 1e-3,
    'adam_epochs': 20000,
}

DATA_PATH = NOTEBOOK_DIR / "data" / "pdebench" / "2D_DarcyFlow_beta1.0_Train.hdf5"

print("PINN Configuration (Pure Adam):")
for key, value in config.items():
    print(f"  {key}: {value}")
print(f"  data_path: {DATA_PATH}")

PINN Configuration (Pure Adam):
  hidden_layers: [32, 32, 32, 32]
  activation: gelu
  resolution: 128
  kappa_index: 0
  n_interior: 2048
  n_boundary: 512
  lambda_pde: 1.0
  lambda_bc: 100.0
  adam_lr: 0.001
  adam_epochs: 20000
  data_path: /workspace/code/07-sciml/data/pdebench/2D_DarcyFlow_beta1.0_Train.hdf5


## Load Permeability Field

Unlike operator learning, PINNs solve for a **single** PDE instance.
We load one permeability field from PDEBench.

In [4]:
import h5py

# Download data if needed
if not DATA_PATH.exists():
    download_pdebench(DATA_PATH)

# Load a single permeability field and reference solution
with h5py.File(DATA_PATH, 'r') as f:
    nu = np.array(f['nu'][config['kappa_index']])
    u_ref = np.array(f['tensor'][config['kappa_index'], 0])

kappa_field = jnp.array(nu)
u_reference = jnp.array(u_ref)

print(f"Loaded permeability field (sample {config['kappa_index']})")
print(f"  kappa shape: {kappa_field.shape}")
print(f"  kappa range: [{float(kappa_field.min()):.4f}, {float(kappa_field.max()):.4f}]")
print(f"  u_ref range: [{float(u_reference.min()):.4f}, {float(u_reference.max()):.4f}]")

Loaded permeability field (sample 0)
  kappa shape: (128, 128)


  kappa range: [0.1000, 1.0000]
  u_ref range: [0.0017, 0.4812]


In [5]:
# Visualize the permeability field
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im0 = axes[0].imshow(np.array(kappa_field).T, origin='lower', cmap='viridis', extent=[0,1,0,1])
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Permeability $\\kappa(x,y)$')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(np.array(u_reference).T, origin='lower', cmap='RdBu_r', extent=[0,1,0,1])
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].set_title('Reference Solution $u(x,y)$')
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
savefig(fig, OUTPUT_DIR / 'pinn_input_reference.png')

Saved: /workspace/code/07-sciml/figures/pinn_input_reference.png


## Random Sampling and Kappa Interpolation

We use **uniform random sampling** for collocation points at each iteration:
- Interior points: uniformly sampled from $[0,1]^2$
- Boundary points: uniformly sampled from $\partial[0,1]^2$

For $\kappa$, we use **piecewise constant interpolation**: given a point $(x,y)$, 
we find the cell it belongs to and return the corresponding $\kappa$ value.

In [6]:
n = config['resolution']

# ============================================================
# Kappa interpolation: piecewise constant lookup
# ============================================================
def get_kappa_piecewise(xy: jnp.ndarray, kappa_field: jnp.ndarray) -> jnp.ndarray:
    """
    Get kappa values via piecewise constant interpolation.
    
    Given points (x, y) in [0,1]^2, find which cell each point belongs to
    and return the corresponding kappa value.
    
    Args:
        xy: Points array of shape (N, 2) with values in [0, 1]
        kappa_field: Kappa values on grid of shape (n, n)
        
    Returns:
        kappa values at each point, shape (N,)
    """
    n_cells = kappa_field.shape[0]
    
    # Map (x, y) to cell indices: floor(x * n), clamped to [0, n-1]
    # For cell-centered data, point (x,y) belongs to cell (floor(x*n), floor(y*n))
    cell_i = jnp.clip(jnp.floor(xy[:, 0] * n_cells).astype(jnp.int32), 0, n_cells - 1)
    cell_j = jnp.clip(jnp.floor(xy[:, 1] * n_cells).astype(jnp.int32), 0, n_cells - 1)
    
    # Lookup kappa values (note: kappa_field is indexed as [j, i] for (x, y))
    return kappa_field[cell_j, cell_i]


# ============================================================
# Random sampling functions (JAX-friendly, vmap-able)
# ============================================================
def sample_interior(key: jax.Array, n_points: int) -> jnp.ndarray:
    """
    Sample n_points uniformly from interior of [0,1]^2.
    
    Args:
        key: JAX PRNG key
        n_points: Number of points to sample
        
    Returns:
        Points array of shape (n_points, 2)
    """
    return random.uniform(key, shape=(n_points, 2), minval=0.0, maxval=1.0)


def sample_boundary(key: jax.Array, n_points: int) -> jnp.ndarray:
    """
    Sample n_points uniformly from boundary of [0,1]^2.
    
    Samples points on all four edges with equal probability.
    
    Args:
        key: JAX PRNG key
        n_points: Number of points to sample
        
    Returns:
        Points array of shape (n_points, 2)
    """
    key1, key2, key3 = random.split(key, 3)
    
    # Sample which edge each point belongs to (0=bottom, 1=top, 2=left, 3=right)
    edges = random.randint(key1, shape=(n_points,), minval=0, maxval=4)
    
    # Sample parameter t in [0, 1] for position along edge
    t = random.uniform(key2, shape=(n_points,), minval=0.0, maxval=1.0)
    
    # Build coordinates based on edge
    # bottom: (t, 0), top: (t, 1), left: (0, t), right: (1, t)
    x = jnp.where(edges == 0, t, 
        jnp.where(edges == 1, t,
        jnp.where(edges == 2, 0.0, 1.0)))
    
    y = jnp.where(edges == 0, 0.0,
        jnp.where(edges == 1, 1.0,
        jnp.where(edges == 2, t, t)))
    
    return jnp.stack([x, y], axis=1)


# ============================================================
# Evaluation grid (for comparing with reference solution)
# ============================================================
cell_centers_1d = (jnp.arange(n) + 0.5) / n
xx, yy = jnp.meshgrid(cell_centers_1d, cell_centers_1d)
xy_eval = jnp.stack([xx.ravel(), yy.ravel()], axis=1)

# Test the sampling functions
test_key = random.PRNGKey(0)
k1, k2 = random.split(test_key)
test_interior = sample_interior(k1, 100)
test_boundary = sample_boundary(k2, 100)
test_kappa = get_kappa_piecewise(test_interior, kappa_field)

print(f"Sampling and interpolation functions defined:")
print(f"  Interior sampling: {test_interior.shape} points in [{float(test_interior.min()):.3f}, {float(test_interior.max()):.3f}]")
print(f"  Boundary sampling: {test_boundary.shape} points")
print(f"  Kappa lookup: {test_kappa.shape} values in [{float(test_kappa.min()):.3f}, {float(test_kappa.max()):.3f}]")
print(f"  Evaluation grid: {xy_eval.shape[0]} points")

Sampling and interpolation functions defined:
  Interior sampling: (100, 2) points in [0.003, 0.996]
  Boundary sampling: (100, 2) points
  Kappa lookup: (100,) values in [0.100, 1.000]
  Evaluation grid: 16384 points


## PINN Network

We use a simple MLP: $(x, y) \mapsto u_\theta(x, y)$

In [7]:
# Initialize network
layer_sizes = [2] + config['hidden_layers'] + [1]
key, subkey = random.split(MASTER_KEY)
params = initialize_mlp(layer_sizes, subkey)

# Activation function
activation_map = {
    'tanh': jax.nn.tanh,
    'relu': jax.nn.relu,
    'gelu': jax.nn.gelu,
}
activation = activation_map[config['activation']]

# Network forward function
def network(params, xy):
    """u_theta(x, y) -> scalar output"""
    return mlp_forward(params, xy, activation)

n_params = count_params(params)
print(f"Network architecture: {layer_sizes}")
print(f"Activation: {config['activation']}")
print(f"Parameters: {n_params:,}")

Network architecture: [2, 32, 32, 32, 32, 1]
Activation: gelu
Parameters: 3,297


## PDE Residual via Automatic Differentiation

The Darcy equation residual:
$$r(x,y) = -\nabla \cdot (\kappa \nabla u) - f = -\kappa \Delta u - \nabla \kappa \cdot \nabla u - f$$

For cell-centered collocation where $\kappa$ is approximately constant:
$$r(x,y) \approx -\kappa \left( \frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2} \right) - f$$

In [8]:
# Note: The residual computation is now defined in the loss function cell above.
# We use vmap to vectorize over collocation points in a JAX-friendly way.

print("Residual computation uses automatic differentiation via JAX hessian()")
print("  r = -kappa * (u_xx + u_yy) - f")
print("  Vectorized with vmap for efficient batch evaluation")

Residual computation uses automatic differentiation via JAX hessian()
  r = -kappa * (u_xx + u_yy) - f
  Vectorized with vmap for efficient batch evaluation


## Loss Function

$$\mathcal{L}(\theta) = \lambda_{\text{PDE}} \frac{1}{N_d}\sum_{i=1}^{N_d} r(x_i, y_i)^2 + \lambda_{\text{BC}} \frac{1}{N_b}\sum_{j=1}^{N_b} u_\theta(x_j, y_j)^2$$

In [9]:
def compute_residual_single(params, xy_single, kappa_single):
    """
    Compute PDE residual at a single point using AD.

    r = -kappa * (u_xx + u_yy) - f
    """
    def u_fn(xy):
        return network(params, xy[None, :])[0, 0]

    # Second derivatives (Hessian)
    hess_u = jax.hessian(u_fn)(xy_single)
    u_xx = hess_u[0, 0]
    u_yy = hess_u[1, 1]

    # Laplacian
    laplacian_u = u_xx + u_yy

    # Residual: -kappa * laplacian(u) - f
    return -kappa_single * laplacian_u - 1.0  # f = 1


# Vectorize over collocation points
compute_residuals = jit(vmap(compute_residual_single, in_axes=(None, 0, 0)))


def loss_fn(params, xy_interior, kappa_interior, xy_boundary):
    """
    Compute total PINN loss for a batch of sampled points.
    
    Args:
        params: Network parameters
        xy_interior: Interior collocation points, shape (n_interior, 2)
        kappa_interior: Kappa values at interior points, shape (n_interior,)
        xy_boundary: Boundary collocation points, shape (n_boundary, 2)
        
    Returns:
        Scalar loss value
    """
    # PDE residual loss
    residuals = compute_residuals(params, xy_interior, kappa_interior)
    loss_pde = jnp.mean(residuals ** 2)

    # Boundary condition loss (u = 0)
    u_boundary = network(params, xy_boundary)
    loss_bc = jnp.mean(u_boundary ** 2)

    # Total loss
    total = config['lambda_pde'] * loss_pde + config['lambda_bc'] * loss_bc
    return total


def loss_fn_with_components(params, xy_interior, kappa_interior, xy_boundary):
    """Compute loss with component breakdown for logging."""
    residuals = compute_residuals(params, xy_interior, kappa_interior)
    loss_pde = jnp.mean(residuals ** 2)

    u_boundary = network(params, xy_boundary)
    loss_bc = jnp.mean(u_boundary ** 2)

    total = config['lambda_pde'] * loss_pde + config['lambda_bc'] * loss_bc
    return total, {'pde': float(loss_pde), 'bc': float(loss_bc)}


# JIT compile the loss and gradient
loss_and_grad = jit(jax.value_and_grad(loss_fn))

print("Loss function defined for batched training")

Loss function defined for batched training


## Training with Adam

We train the PINN using Adam optimizer with random sampling of collocation points at each iteration.

In [10]:
import optax

# Adam optimizer
optimizer = optax.adam(config['adam_lr'])
opt_state = optimizer.init(params)

@jit
def adam_step(params, opt_state, xy_interior, kappa_interior, xy_boundary):
    """Single Adam update step."""
    loss, grads = loss_and_grad(params, xy_interior, kappa_interior, xy_boundary)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss


def train_epoch(params, opt_state, key):
    """
    Run one training epoch with fresh random samples.
    
    Returns:
        Updated params, opt_state, and loss value
    """
    key1, key2 = random.split(key)
    
    # Sample fresh collocation points
    xy_interior = sample_interior(key1, config['n_interior'])
    xy_boundary = sample_boundary(key2, config['n_boundary'])
    
    # Get kappa values at interior points
    kappa_interior = get_kappa_piecewise(xy_interior, kappa_field)
    
    # Adam step
    params, opt_state, loss = adam_step(params, opt_state, xy_interior, kappa_interior, xy_boundary)
    
    return params, opt_state, loss


print("Training setup: Pure Adam")
print(f"  Learning rate: {config['adam_lr']}")
print(f"  Iterations: {config['adam_epochs']}")
print(f"  Interior points per batch: {config['n_interior']}")
print(f"  Boundary points per batch: {config['n_boundary']}")

Training setup: Pure Adam
  Learning rate: 0.001
  Iterations: 20000
  Interior points per batch: 2048
  Boundary points per batch: 512


In [11]:
# Training with Adam (20k iterations)
print("\n" + "=" * 70)
print("TRAINING PINN WITH ADAM")
print("=" * 70)

# Training key
key, train_key = random.split(MASTER_KEY)

# Check initial loss with a sample batch
k1, k2, train_key = random.split(train_key, 3)
xy_int_init = sample_interior(k1, config['n_interior'])
xy_bnd_init = sample_boundary(k2, config['n_boundary'])
kappa_int_init = get_kappa_piecewise(xy_int_init, kappa_field)
init_loss, init_comp = loss_fn_with_components(params, xy_int_init, kappa_int_init, xy_bnd_init)
print(f"Initial loss: {init_loss:.6e} (PDE: {init_comp['pde']:.6e}, BC: {init_comp['bc']:.6e})")

start_time = time()
loss_history = []
pde_loss_history = []
bc_loss_history = []
print_every = config['adam_epochs'] // 10  # Print 10 times during training

for epoch in range(config['adam_epochs']):
    train_key, epoch_key = random.split(train_key)
    params, opt_state, loss = train_epoch(params, opt_state, epoch_key)
    loss_history.append(float(loss))
    
    if (epoch + 1) % print_every == 0 or epoch == 0:
        # Get detailed loss breakdown
        k1, k2 = random.split(epoch_key)
        xy_int = sample_interior(k1, config['n_interior'])
        xy_bnd = sample_boundary(k2, config['n_boundary'])
        kappa_int = get_kappa_piecewise(xy_int, kappa_field)
        _, comp = loss_fn_with_components(params, xy_int, kappa_int, xy_bnd)
        pde_loss_history.append(comp['pde'])
        bc_loss_history.append(comp['bc'])
        
        elapsed = time() - start_time
        print(f"Iter {epoch+1:5d} | Loss: {loss:.6e} | PDE: {comp['pde']:.3e} | BC: {comp['bc']:.3e} | Time: {elapsed:.1f}s")

training_time = time() - start_time

# Final evaluation
u_pred_flat = network(params, xy_eval)
u_pred = u_pred_flat.reshape(n, n)
rel_error = compute_l2_error(np.array(u_pred), np.array(u_reference))

print("-" * 70)
print(f"Training complete in {training_time:.1f}s")
print(f"  Final loss: {loss_history[-1]:.6e}")
print(f"  Relative L2 Error vs PDEBench: {rel_error:.4f} ({rel_error*100:.2f}%)")


TRAINING PINN WITH ADAM


Initial loss: 1.088438e+00 (PDE: 1.001944e+00, BC: 8.649447e-04)


Iter     1 | Loss: 1.091638e+00 | PDE: 1.001e+00 | BC: 2.966e-04 | Time: 0.8s


Iter  2000 | Loss: 2.607433e-01 | PDE: 2.447e-01 | BC: 1.434e-04 | Time: 20.4s


Iter  4000 | Loss: 1.615909e-01 | PDE: 1.499e-01 | BC: 1.311e-04 | Time: 39.9s


...

Iter 14000 | Loss: 8.592290e-02 | PDE: 7.748e-02 | BC: 1.238e-05 | Time: 139.4s


Iter 16000 | Loss: 5.857540e-02 | PDE: 5.736e-02 | BC: 2.156e-05 | Time: 159.6s


Iter 18000 | Loss: 6.642552e-02 | PDE: 6.628e-02 | BC: 3.427e-05 | Time: 179.4s


Iter 20000 | Loss: 5.287541e-02 | PDE: 5.088e-02 | BC: 1.456e-05 | Time: 199.1s


----------------------------------------------------------------------
Training complete in 199.1s
  Final loss: 5.287541e-02
  Relative L2 Error vs PDEBench: 0.3788 (37.88%)


In [12]:
# Save training history (sampled at regular intervals for TikZ plots in slides)
# Sample at powers of 2 and regular intervals for good coverage
sample_iters = [0] + list(range(1999, config['adam_epochs'], 2000)) + [config['adam_epochs']-1]
sample_iters = sorted(set(sample_iters))

history_df = pd.DataFrame({
    'iteration': sample_iters,
    'loss': [loss_history[i] for i in sample_iters]
})
history_df.to_csv(OUTPUT_DIR / 'pinn_training_history.csv', index=False)
print(f"Saved: {OUTPUT_DIR / 'pinn_training_history.csv'}")
print(history_df)

Saved: /workspace/code/07-sciml/figures/pinn_training_history.csv
    iteration      loss
0           0  1.091638
1        1999  0.260743
2        3999  0.161591
3        5999  0.131035
4        7999  0.112820
5        9999  0.092667
6       11999  0.075794
7       13999  0.085923
8       15999  0.058575
9       17999  0.066426
10      19999  0.052875


In [13]:
# Plot training convergence
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(loss_history, 'b-', linewidth=1, alpha=0.7)

# Add smoothed version
window = min(500, len(loss_history) // 20)
if window > 1:
    smoothed = pd.Series(loss_history).rolling(window=window, center=True).mean()
    ax.semilogy(smoothed, 'r-', linewidth=2, label=f'Smoothed (window={window})')
    ax.legend()

ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.set_title(f'PINN Training ({config["adam_epochs"]} iterations)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
savefig(fig, OUTPUT_DIR / 'pinn_convergence.png')

Saved: /workspace/code/07-sciml/figures/pinn_convergence.png


## Evaluate Solution

In [14]:
# Evaluation was done at end of training
# u_pred is already computed as u_pred_flat.reshape(n, n)
# rel_error is already computed

print(f"\nEvaluation Results:")
print(f"  Relative L2 Error: {rel_error:.4f} ({rel_error*100:.2f}%)")
print(f"  u_pred range: [{float(u_pred.min()):.4f}, {float(u_pred.max()):.4f}]")
print(f"  u_ref range:  [{float(u_reference.min()):.4f}, {float(u_reference.max()):.4f}]")


Evaluation Results:
  Relative L2 Error: 0.3788 (37.88%)
  u_pred range: [-0.0148, 0.3986]
  u_ref range:  [0.0017, 0.4812]


In [15]:
# Visualize solution comparison
fig = plot_solution_comparison(
    np.array(kappa_field),
    np.array(u_reference),
    np.array(u_pred),
    title='PINN Solution vs Reference'
)
savefig(fig, OUTPUT_DIR / 'pinn_solution_comparison.png')

# Save individual figures for slides (no trimming needed)
u_vmin = min(float(u_reference.min()), float(u_pred.min()))
u_vmax = max(float(u_reference.max()), float(u_pred.max()))
error = np.abs(np.array(u_pred) - np.array(u_reference))

save_individual_figure(np.array(kappa_field), 'Permeability $\\kappa$',
                       OUTPUT_DIR / 'pinn_kappa.png', cmap='viridis')
save_individual_figure(np.array(u_reference), 'Reference $u$ (PDEBench)',
                       OUTPUT_DIR / 'pinn_reference.png', cmap='RdBu_r',
                       vmin=u_vmin, vmax=u_vmax)
save_individual_figure(np.array(u_pred), f'PINN $u_\\theta$ ({rel_error*100:.1f}%)',
                       OUTPUT_DIR / 'pinn_prediction.png', cmap='RdBu_r',
                       vmin=u_vmin, vmax=u_vmax)
save_individual_figure(error, '$|u_\\theta - u_{ref}|$',
                       OUTPUT_DIR / 'pinn_error.png', cmap='hot')

Saved: /workspace/code/07-sciml/figures/pinn_solution_comparison.png
Saved: /workspace/code/07-sciml/figures/pinn_kappa.png


Saved: /workspace/code/07-sciml/figures/pinn_reference.png


Saved: /workspace/code/07-sciml/figures/pinn_prediction.png
Saved: /workspace/code/07-sciml/figures/pinn_error.png


## Spectral Analysis: Understanding PINN Limitations

PINNs suffer from **spectral bias** - they learn low-frequency components faster than high-frequency ones.
This is particularly problematic when the permeability $\kappa$ has sharp features.

### What is Spectral Bias?

Neural networks with smooth activations (like tanh, GELU) tend to learn low-frequency functions first. This means:
1. The overall shape of the solution is learned quickly
2. Fine details and sharp transitions are learned slowly (or never)
3. Error concentrates at high frequencies

In [16]:
# Compare spectra
error = np.array(u_pred) - np.array(u_reference)

# 2D FFT
u_ref_fft = np.abs(np.fft.fftshift(np.fft.fft2(np.array(u_reference))))
u_pred_fft = np.abs(np.fft.fftshift(np.fft.fft2(np.array(u_pred))))
error_fft = np.abs(np.fft.fftshift(np.fft.fft2(error)))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Reference spectrum
im0 = axes[0].imshow(np.log10(u_ref_fft + 1e-10), cmap='viridis')
axes[0].set_title('Reference Spectrum (log)')
plt.colorbar(im0, ax=axes[0])

# PINN spectrum
im1 = axes[1].imshow(np.log10(u_pred_fft + 1e-10), cmap='viridis')
axes[1].set_title('PINN Spectrum (log)')
plt.colorbar(im1, ax=axes[1])

# Error spectrum
im2 = axes[2].imshow(np.log10(error_fft + 1e-10), cmap='hot')
axes[2].set_title('Error Spectrum (log)')
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
savefig(fig, OUTPUT_DIR / 'pinn_spectral_analysis.png')

print("Note: Error is concentrated at high frequencies (edges of spectrum)")
print("This is the 'spectral bias' - PINNs struggle with high-frequency content.")

Saved: /workspace/code/07-sciml/figures/pinn_spectral_analysis.png
Note: Error is concentrated at high frequencies (edges of spectrum)
This is the 'spectral bias' - PINNs struggle with high-frequency content.


In [17]:
# Radially averaged power spectrum for quantitative analysis
def radial_profile(data):
    """Compute radially averaged profile of 2D data."""
    center = np.array(data.shape) // 2
    y, x = np.ogrid[:data.shape[0], :data.shape[1]]
    r = np.sqrt((x - center[1])**2 + (y - center[0])**2).astype(int)

    # Bin by radius
    r_max = min(center)
    tbin = np.bincount(r.ravel(), data.ravel())
    nr = np.bincount(r.ravel())
    radialprofile = tbin / (nr + 1e-10)
    return radialprofile[:r_max]

# Compute radial profiles
r_ref = radial_profile(u_ref_fft)
r_pred = radial_profile(u_pred_fft)
r_error = radial_profile(error_fft)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Power spectra
freqs = np.arange(len(r_ref))
axes[0].semilogy(freqs, r_ref, 'b-', linewidth=2, label='Reference')
axes[0].semilogy(freqs, r_pred, 'r--', linewidth=2, label='PINN')
axes[0].set_xlabel('Frequency (radial)')
axes[0].set_ylabel('Spectral Power')
axes[0].set_title('Radially Averaged Power Spectrum')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Relative error by frequency
rel_spectral_error = np.abs(r_pred - r_ref) / (r_ref + 1e-10)
axes[1].semilogy(freqs, rel_spectral_error, 'k-', linewidth=2)
axes[1].axvline(x=len(freqs)//4, color='r', linestyle='--', label='Low/Mid freq boundary')
axes[1].axvline(x=len(freqs)//2, color='orange', linestyle='--', label='Mid/High freq boundary')
axes[1].set_xlabel('Frequency (radial)')
axes[1].set_ylabel('Relative Spectral Error')
axes[1].set_title('Error vs Frequency (Spectral Bias)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
savefig(fig, OUTPUT_DIR / 'pinn_spectral_bias.png')

# Save spectral data to CSV
spectral_df = pd.DataFrame({
    'frequency': freqs,
    'reference_power': r_ref,
    'pinn_power': r_pred,
    'error_power': r_error,
    'relative_error': rel_spectral_error
})
spectral_df.to_csv(OUTPUT_DIR / 'pinn_spectral_data.csv', index=False)
print(f"Saved: {OUTPUT_DIR / 'pinn_spectral_data.csv'}")

Saved: /workspace/code/07-sciml/figures/pinn_spectral_bias.png
Saved: /workspace/code/07-sciml/figures/pinn_spectral_data.csv


## Summary

### Key Takeaways

1. **PINNs are data-free**: They use PDE physics as supervision
2. **Loss balancing is critical**: The $\lambda_{BC}$ weight needs tuning
3. **Adam optimization**: 20k iterations with random sampling
4. **Piecewise constant interpolation**: Efficient kappa lookup without smooth interpolation
5. **Spectral bias**: PINNs struggle with high-frequency content

### When to Use PINNs

**Good for:**
- Inverse problems (unknown PDE parameters)
- Sparse/noisy observations
- Problems where classical solvers struggle

**Limited for:**
- High-accuracy forward problems (classical solvers are better)
- Problems with sharp features or high-frequency content
- Operator learning (solving many instances - use FNO/DeepONet)

### Next Steps

See `operator-learning.ipynb` for neural operators (FNO, DeepONet) that:
- Learn to solve **families** of PDEs
- Amortize computation across instances
- Achieve instant inference after training

In [18]:
# Final summary
print("\n" + "=" * 60)
print("PINN RESULTS SUMMARY")
print("=" * 60)
print(f"  Network: {layer_sizes}")
print(f"  Parameters: {n_params:,}")
print(f"  Training: Adam ({config['adam_epochs']} iterations, lr={config['adam_lr']})")
print(f"  Training time: {training_time:.1f}s")
print(f"  Final loss: {loss_history[-1]:.6e}")
print(f"  Relative L2 Error vs PDEBench: {rel_error:.4f} ({rel_error*100:.2f}%)")
print("=" * 60)
print("\nSAVED FILES")
print("=" * 60)
for f in sorted(OUTPUT_DIR.glob('pinn_*')):
    print(f"  {f.name}")
print("=" * 60)

# Save summary to JSON for slides
import json
summary = {
    'method': 'pinn',
    'network': layer_sizes,
    'n_params': n_params,
    'adam_epochs': config['adam_epochs'],
    'training_time_s': training_time,
    'final_loss': float(loss_history[-1]),
    'rel_l2_error': float(rel_error),
}
with open(OUTPUT_DIR / 'pinn_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved summary: {OUTPUT_DIR / 'pinn_summary.json'}")


PINN RESULTS SUMMARY
  Network: [2, 32, 32, 32, 32, 1]
  Parameters: 3,297
  Training: Adam (20000 iterations, lr=0.001)
  Training time: 199.1s
  Final loss: 5.287541e-02
  Relative L2 Error vs PDEBench: 0.3788 (37.88%)

SAVED FILES
  pinn_convergence.png
  pinn_error.png
  pinn_input_reference.png
  pinn_kappa.png
  pinn_prediction.png
  pinn_reference.png
  pinn_solution_comparison.png
  pinn_spectral_analysis.png
  pinn_spectral_bias.png
  pinn_spectral_data.csv
  pinn_training_history.csv
  pinn_training_results.csv

Saved summary: /workspace/code/07-sciml/figures/pinn_summary.json
